# LeetCode 2540 — Minimum Common Value bằng Python

Link đề: https://leetcode.com/problems/minimum-common-value/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Two Pointers, Binary Search

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def getCommon(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **số nguyên nhỏ nhất xuất hiện trong cả hai mảng** đã sắp xếp không giảm; nếu không có thì trả về `-1`.
2. Biết dùng `set` để lấy **giao** của hai tập hợp, và biết vì sao mảng đã sắp xếp cho phép giải **không cần bộ nhớ phụ**.
3. Hiểu kỹ thuật **hai con trỏ** (two pointers) trên hai mảng đã sắp xếp: tiến con trỏ của mảng có giá trị nhỏ hơn.
4. Viết 4 cách: duyệt lồng nhau, giao hai `set`, hai con trỏ, và tìm kiếm nhị phân `bisect`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: lấy giá trị chung **lớn nhất**, trả `0` thay vì `-1`, tiến cả hai con trỏ khi chưa bằng nhau, và chỉ so sánh phần tử đầu mảng.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2` đã được **sắp xếp theo thứ tự không giảm**. Trả về **số nguyên nhỏ nhất** có mặt ở **cả hai** mảng. Nếu hai mảng **không có giá trị chung**, trả về `-1`.

| Ví dụ | nums1 | nums2 | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[1,2,3]` | `[2,4]` | `2` | `2` là giá trị chung nhỏ nhất |
| 2 | `[1,2,3,6]` | `[2,3,4,5]` | `2` | chung có `2` và `3`; nhỏ nhất là `2` |

Constraints: `1 <= nums1.length, nums2.length <= 10^5`; `1 <= nums1[i], nums2[j] <= 10^9`; cả hai mảng được sắp xếp theo thứ tự không giảm.

In [1]:
examples = [([1, 2, 3], [2, 4], 2), ([1, 2, 3, 6], [2, 3, 4, 5], 2)]
for a, b, expected in examples:
    common = sorted(set(a) & set(b))
    ans = common[0] if common else -1
    print(f"nums1={a}, nums2={b}: các giá trị chung={common} -> {ans} (mong đợi {expected})")

nums1=[1, 2, 3], nums2=[2, 4]: các giá trị chung=[2] -> 2 (mong đợi 2)
nums1=[1, 2, 3, 6], nums2=[2, 3, 4, 5]: các giá trị chung=[2, 3] -> 2 (mong đợi 2)


## 3. Kiến thức nền

### 3.1. `set` và phép giao `&`

`set` là tập hợp không có phần tử trùng, kiểm tra thuộc `x in s` trung bình `O(1)`. Phép giao `a & b` (hoặc `a.intersection(b)`) cho các phần tử có ở **cả hai** tập.

In [2]:
a, b = {1, 2, 3, 6}, {2, 3, 4, 5}
print(a & b, a.intersection(b))
print(2 in a, 7 in a)

{2, 3} {2, 3}
True False


### 3.2. `min` trên tập có thể rỗng

`min(s)` trên tập **rỗng** gây `ValueError`. Dùng `min(s, default=-1)` để tránh lỗi và trả `-1` khi rỗng. Đây chính là đáp án mặc định của bài.

In [3]:
print(min({3, 2}, default=-1))
print(min(set(), default=-1))
try:
    min(set())
except ValueError as e:
    print("ValueError:", e)

2
-1
ValueError: min() iterable argument is empty


### 3.3. Mảng đã sắp xếp và hai con trỏ

Với hai mảng sắp xếp `a`, `b`, đặt `i = 0`, `j = 0`. Nếu `a[i] < b[j]`, phần tử `a[i]` **không thể** bằng bất kỳ `b[j'] >= b[j]`, nên bỏ `a[i]` (tăng `i`). Nếu `a[i] > b[j]` thì tăng `j`. Nếu bằng nhau: đã tìm được.

In [4]:
a, b = [1, 2, 3], [2, 4]
i = j = 0
while i < len(a) and j < len(b):
    print(f"i={i}, j={j}: a[i]={a[i]}, b[j]={b[j]}")
    if a[i] == b[j]:
        print("chung:", a[i]); break
    elif a[i] < b[j]:
        i += 1
    else:
        j += 1

i=0, j=0: a[i]=1, b[j]=2
i=1, j=0: a[i]=2, b[j]=2
chung: 2


### 3.4. `bisect_left` — tìm một giá trị trong mảng sắp xếp

`bisect_left(arr, x)` trả vị trí `p` đầu tiên có `arr[p] >= x`. Giá trị `x` có mặt trong `arr` khi `p < len(arr)` và `arr[p] == x`. Độ phức tạp `O(log n)`.

In [5]:
from bisect import bisect_left
arr = [2, 4, 4, 7]
for x in (4, 5, 1, 8):
    p = bisect_left(arr, x)
    print(x, "-> p =", p, "| có mặt:", p < len(arr) and arr[p] == x)

4 -> p = 1 | có mặt: True
5 -> p = 3 | có mặt: False
1 -> p = 0 | có mặt: False
8 -> p = 4 | có mặt: False


### 3.5. Phần tử đầu tiên thỏa điều kiện: `next(...)`

`next(gen, default)` lấy phần tử đầu tiên của generator, hoặc `default` nếu rỗng. Vì mảng sắp xếp tăng, duyệt từ trái sang phải thì phần tử chung đầu tiên gặp được là nhỏ nhất.

In [6]:
a, b = [1, 2, 3, 6], {2, 3, 4, 5}
print(next((x for x in a if x in b), -1))
print(next((x for x in [1, 9] if x in b), -1))

2
-1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `nums1 = [1,2,3,6]`, `nums2 = [2,3,4,5]`:

- `1`: có trong `nums2`? Không.
- `2`: có trong `nums2`? Có. Vì `nums1` đã sắp xếp và ta duyệt từ nhỏ đến lớn, đây là giá trị chung **đầu tiên** nên cũng là **nhỏ nhất**.

Hoàn toàn tương tự với `nums1 = [1,2,3]`, `nums2 = [4,5]`: không giá trị nào của `nums1` có trong `nums2` nên đáp án `-1`.

**Nhận xét:** nhờ sắp xếp, ta không cần tìm hết các giá trị chung rồi lấy `min`; chỉ cần dừng ở giá trị chung đầu tiên.

In [7]:
nums1, nums2 = [1, 2, 3, 6], [2, 3, 4, 5]
for x in nums1:
    print(x, "có trong nums2?", x in nums2)

1 có trong nums2? False
2 có trong nums2? True
3 có trong nums2? True
6 có trong nums2? False


## 5. Cách 1 — Brute Force / baseline: duyệt lồng nhau

**Ý tưởng:** với mỗi `x` của `nums1` (từ nhỏ đến lớn), duyệt `nums2` để tìm `x`. Giá trị chung đầu tiên gặp được là nhỏ nhất.

In [8]:
from typing import List

class SolutionBrute:
    def getCommon(self, nums1: List[int], nums2: List[int]) -> int:
        for x in nums1:
            for y in nums2:
                if x == y:
                    return x
        return -1

In [9]:
sol = SolutionBrute()
print(sol.getCommon([1, 2, 3], [2, 4]))             # 2
print(sol.getCommon([1, 2, 3, 6], [2, 3, 4, 5]))    # 2
print(sol.getCommon([1, 2, 3], [4, 5]))             # -1

2
2
-1


**Độ phức tạp:** Time `O(n * m)` với `n = len(nums1)`, `m = len(nums2)`; với `10^5 x 10^5` là `10^10` phép so sánh, quá chậm. Space `O(1)`. **Nút thắt:** không tận dụng việc hai mảng đã sắp xếp.

## 6. Ý tưởng thuật toán / bước đột phá

Mảng đã sắp xếp nên ta có thể "đi song song" như bước trộn (merge) của merge sort:

- Hai con trỏ `i` (trên `nums1`) và `j` (trên `nums2`).
- Nếu `nums1[i] == nums2[j]`: trả về giá trị này (đã là nhỏ nhất vì mọi giá trị nhỏ hơn đã bị loại).
- Nếu `nums1[i] < nums2[j]`: `nums1[i]` nhỏ hơn mọi phần tử còn lại của `nums2`, nên **không thể là giá trị chung**, tăng `i`.
- Ngược lại tăng `j`.

**Quy tắc:** *tiến con trỏ của mảng có phần tử nhỏ hơn; gặp bằng nhau thì dừng.*

**Invariant:** mọi giá trị chung nhỏ hơn `min(nums1[i], nums2[j])` (nếu có) đã được loại trừ; do đó giá trị chung đầu tiên tìm được chính là nhỏ nhất.

## 7. Các cách tiếp theo

### Cách 2 — Giao hai `set`

Chuyển cả hai mảng thành `set`, lấy giao, rồi lấy `min` (với `default=-1`).

In [10]:
from typing import List

class SolutionSet:
    def getCommon(self, nums1: List[int], nums2: List[int]) -> int:
        return min(set(nums1) & set(nums2), default=-1)

In [11]:
sol = SolutionSet()
print(sol.getCommon([1, 2, 3], [2, 4]))             # 2
print(sol.getCommon([1, 2, 3, 6], [2, 3, 4, 5]))    # 2
print(sol.getCommon([1, 2, 3], [4, 5]))             # -1

2
2
-1


**Độ phức tạp:** Time `O(n + m)` trung bình, Space `O(n + m)` cho hai tập. Không tận dụng thứ tự sắp xếp nhưng ngắn gọn.

### Cách 3 — Hai con trỏ (cách nộp bài)

Đúng như mục 6, không dùng bộ nhớ phụ.

In [12]:
from typing import List

class SolutionTwoPointers:
    def getCommon(self, nums1: List[int], nums2: List[int]) -> int:
        i = j = 0
        while i < len(nums1) and j < len(nums2):
            if nums1[i] == nums2[j]:
                return nums1[i]
            if nums1[i] < nums2[j]:
                i += 1
            else:
                j += 1
        return -1

In [13]:
sol = SolutionTwoPointers()
print(sol.getCommon([1, 2, 3], [2, 4]))             # 2
print(sol.getCommon([1, 2, 3, 6], [2, 3, 4, 5]))    # 2
print(sol.getCommon([1, 2, 3], [4, 5]))             # -1

2
2
-1


**Độ phức tạp:** Time `O(n + m)` (mỗi bước tăng một con trỏ, tối đa `n + m` bước), Space `O(1)`.

### Cách 4 — Tìm kiếm nhị phân bằng `bisect`

Duyệt `x` của `nums1` từ nhỏ đến lớn; dùng `bisect_left` tìm `x` trong `nums2`. Phù hợp khi một mảng **rất ngắn** so với mảng còn lại (duyệt mảng ngắn, tìm nhị phân trong mảng dài).

In [14]:
from bisect import bisect_left
from typing import List

class SolutionBisect:
    def getCommon(self, nums1: List[int], nums2: List[int]) -> int:
        if len(nums1) > len(nums2):
            nums1, nums2 = nums2, nums1                 # duyệt mảng ngắn hơn
        for x in nums1:
            p = bisect_left(nums2, x)
            if p < len(nums2) and nums2[p] == x:
                return x
        return -1

In [15]:
sol = SolutionBisect()
print(sol.getCommon([1, 2, 3], [2, 4]))             # 2
print(sol.getCommon([1, 2, 3, 6], [2, 3, 4, 5]))    # 2
print(sol.getCommon([1, 2, 3], [4, 5]))             # -1

2
2
-1


**Độ phức tạp:** Time `O(min(n, m) * log(max(n, m)))`, Space `O(1)`. Lưu ý: sau khi hoán đổi để duyệt mảng ngắn, giá trị chung đầu tiên theo thứ tự của mảng ngắn vẫn là nhỏ nhất vì mảng ngắn cũng sắp xếp tăng dần.

## 8. TRACE từng bước

Trace Cách 3 với `nums1 = [1,2,3,6]`, `nums2 = [2,3,4,5]`:

| Bước | `i` | `j` | `nums1[i]` | `nums2[j]` | Quyết định |
|---|---|---|---|---|---|
| 1 | 0 | 0 | 1 | 2 | `1 < 2` nên `i = 1` |
| 2 | 1 | 0 | 2 | 2 | bằng nhau nên trả `2` |

Với `nums1 = [1,2,3]`, `nums2 = [4,5]`: `i` tăng đến `3` (hết `nums1`) mà `j` vẫn `0`, trả `-1`.

In [16]:
def trace_two_pointers(a, b):
    print("nums1 =", a, "| nums2 =", b)
    i = j = 0
    step = 0
    while i < len(a) and j < len(b):
        step += 1
        print(f"  bước {step}: i={i}, j={j}, a[i]={a[i]}, b[j]={b[j]}", end=" -> ")
        if a[i] == b[j]:
            print("bằng nhau, trả về", a[i])
            return a[i]
        if a[i] < b[j]:
            print("a[i] < b[j]: i += 1")
            i += 1
        else:
            print("a[i] > b[j]: j += 1")
            j += 1
    print("  hết một mảng -> -1")
    return -1

trace_two_pointers([1, 2, 3, 6], [2, 3, 4, 5])
trace_two_pointers([1, 2, 3], [4, 5])
trace_two_pointers([5, 6, 7], [1, 2, 5, 7])

nums1 = [1, 2, 3, 6] | nums2 = [2, 3, 4, 5]
  bước 1: i=0, j=0, a[i]=1, b[j]=2 -> a[i] < b[j]: i += 1
  bước 2: i=1, j=0, a[i]=2, b[j]=2 -> bằng nhau, trả về 2
nums1 = [1, 2, 3] | nums2 = [4, 5]
  bước 1: i=0, j=0, a[i]=1, b[j]=4 -> a[i] < b[j]: i += 1
  bước 2: i=1, j=0, a[i]=2, b[j]=4 -> a[i] < b[j]: i += 1
  bước 3: i=2, j=0, a[i]=3, b[j]=4 -> a[i] < b[j]: i += 1
  hết một mảng -> -1
nums1 = [5, 6, 7] | nums2 = [1, 2, 5, 7]
  bước 1: i=0, j=0, a[i]=5, b[j]=1 -> a[i] > b[j]: j += 1
  bước 2: i=0, j=1, a[i]=5, b[j]=2 -> a[i] > b[j]: j += 1
  bước 3: i=0, j=2, a[i]=5, b[j]=5 -> bằng nhau, trả về 5


5

## 9. Code nộp LeetCode

Hai con trỏ:

In [17]:
from typing import List

class Solution:
    def getCommon(self, nums1: List[int], nums2: List[int]) -> int:
        i = j = 0
        while i < len(nums1) and j < len(nums2):
            if nums1[i] == nums2[j]:
                return nums1[i]
            if nums1[i] < nums2[j]:
                i += 1
            else:
                j += 1
        return -1

In [18]:
print(Solution().getCommon([1, 2, 3], [2, 4]))
print(Solution().getCommon([1, 2, 3, 6], [2, 3, 4, 5]))
print(Solution().getCommon([1, 2, 3], [4, 5]))

2
2
-1


## 10. Vì sao thuật toán đúng

1. **Loại trừ đúng.** Khi `nums1[i] < nums2[j]`, vì `nums2` không giảm nên mọi phần tử `nums2[j'] (j' >= j)` đều `>= nums2[j] > nums1[i]`. Còn các phần tử `nums2[j'] (j' < j)` đã bị loại trước đó. Do vậy `nums1[i]` không có bản sao ở `nums2` và có thể bỏ. Tương tự khi `nums1[i] > nums2[j]`.
2. **Tìm thấy là nhỏ nhất.** Mọi giá trị chung `v` thỏa `v < nums1[i]` hoặc `v < nums2[j]` đều đã bị loại ở các bước trước (invariant). Khi `nums1[i] == nums2[j] = v`, không còn giá trị chung nào nhỏ hơn `v` nên `v` là nhỏ nhất.
3. **Kết thúc.** Mỗi vòng tăng `i` hoặc `j` hoặc dừng nên dừng sau tối đa `n + m` vòng.
4. **Không có giá trị chung.** Nếu một mảng hết trước khi gặp bằng nhau, mọi phần tử đã bị loại hết nên không có giá trị chung: trả `-1`.
5. **Trùng lặp.** Phần tử lặp lại trong một mảng không ảnh hưởng: gặp bằng nhau lần đầu là dừng.
6. **Cách `set`.** Giao hai tập chứa đúng các giá trị có ở cả hai mảng; `min` của nó là đáp án; rỗng thì `-1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt lồng nhau | `O(n * m)` | `O(1)` |
| 2. Giao hai `set` | `O(n + m)` trung bình | `O(n + m)` |
| 3. Hai con trỏ | `O(n + m)` | `O(1)` |
| 4. `bisect` | `O(min(n, m) * log(max(n, m)))` | `O(1)` |

`n = len(nums1)`, `m = len(nums2)`, đều `<= 10^5`. Cách 3 tối ưu cả thời gian và bộ nhớ; Cách 4 tốt khi một mảng ngắn hơn rất nhiều.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Mỗi kỳ vọng đều tính tay (ghi chú bên cạnh).

In [19]:
sols = [SolutionBrute(), SolutionSet(), SolutionTwoPointers(), SolutionBisect(), Solution()]
cases = [
    ([1, 2, 3], [2, 4], 2),                    # ví dụ 1
    ([1, 2, 3, 6], [2, 3, 4, 5], 2),           # ví dụ 2
    ([1, 2, 3], [4, 5], -1),                   # không chung
    ([1], [1], 1),                             # cả hai chỉ một phần tử, bằng nhau
    ([1], [2], -1),                            # cả hai chỉ một phần tử, khác nhau
    ([1, 1, 1], [1], 1),                       # lặp lại
    ([1, 3, 5], [2, 4, 6], -1),                # đan xen, không chung
    ([5, 6, 7], [1, 2, 5, 7], 5),              # chung 5 và 7, nhỏ nhất 5
    ([1, 2, 2, 3], [2, 2, 3, 4], 2),           # chung 2 và 3
    ([10**9], [10**9], 10**9),                 # biên trên
    ([1], [10**9], -1),                        # hai biên khác nhau
    ([3, 4, 5], [1, 2, 3, 4, 5], 3),           # mảng 1 là đoạn con của mảng 2
    ([2, 2, 3], [3, 3, 4], 3),                 # chung duy nhất là 3
    ([4, 8, 12], [1, 5, 9, 12], 12),           # chung ở cuối
]
for a, b, want in cases:
    for s in sols:
        got = s.getCommon(a[:], b[:])
        assert got == want, (type(s).__name__, a, b, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 14


Phản ví dụ: với `[1,2,3,6]` và `[2,3,4,5]` các giá trị chung là `{2, 3}`; nếu lấy lớn nhất ra `3` (sai), đáp án đúng `2`.

In [20]:
assert Solution().getCommon([1, 2, 3, 6], [2, 3, 4, 5]) == 2
assert max(set([1, 2, 3, 6]) & set([2, 3, 4, 5])) == 3 != 2
print("phản ví dụ OK")

phản ví dụ OK


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh hai mảng **đã sắp xếp không giảm** đúng constraints. Với test nhỏ: độ dài `1..15`, giá trị `1..30` (nhiều giá trị chung/trùng); với test lớn: độ dài tới `10^5`, giá trị tới `10^9` (cách `Brute` quá chậm nên bỏ).

In [21]:
import random
random.seed(2540)

def gen(n, lo, hi):
    return sorted(random.randint(lo, hi) for _ in range(n))

def oracle(a, b):
    common = set(a) & set(b)
    return min(common) if common else -1

for t in range(3000):
    a = gen(random.randint(1, 15), 1, random.choice([10, 30, 100]))
    b = gen(random.randint(1, 15), 1, random.choice([10, 30, 100]))
    ans = [s.getCommon(a[:], b[:]) for s in sols]
    assert len(set(ans)) == 1 and ans[0] == oracle(a, b), (a, b, ans)

fast = [SolutionSet(), SolutionTwoPointers(), SolutionBisect(), Solution()]
for t in range(40):
    n, m = random.choice([(10**5, 10**5), (10**5, 50), (3, 10**5), (2000, 3000)])
    hi = random.choice([10**9, 10**6, 10**3])
    a, b = gen(n, 1, hi), gen(m, 1, hi)
    assert 1 <= len(a) <= 10**5 and 1 <= len(b) <= 10**5
    assert all(1 <= x <= 10**9 for x in a + b)
    ans = [s.getCommon(a, b) for s in fast]
    assert len(set(ans)) == 1 and ans[0] == oracle(a, b), (n, m, ans)
print("3000 test nhỏ (5 cách) + 40 test lớn (4 cách nhanh): khớp nhau và khớp oracle")

3000 test nhỏ (5 cách) + 40 test lớn (4 cách nhanh): khớp nhau và khớp oracle


### 12.3. Đo thời gian trên dữ liệu lớn

So sánh hai con trỏ và `set` trên hai mảng `10^5` phần tử không có giá trị chung (trường hợp xấu nhất: phải duyệt hết).

In [22]:
import time
a = list(range(1, 200001, 2))                  # số lẻ 1..199999: 100000 phần tử
b = list(range(2, 200002, 2))                  # số chẵn 2..200000: 100000 phần tử
assert len(a) == len(b) == 100000
for name, s in (("hai con trỏ", SolutionTwoPointers()), ("set", SolutionSet()), ("bisect", SolutionBisect())):
    t0 = time.perf_counter()
    r = s.getCommon(a, b)
    dt = time.perf_counter() - t0
    assert r == -1
    print(f"{name:12s}: kết quả {r}, {dt * 1000:.1f} ms")

hai con trỏ : kết quả -1, 40.8 ms
set         : kết quả -1, 14.0 ms
bisect      : kết quả -1, 37.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lấy giá trị chung **lớn nhất**

In [23]:
def sai_lay_max(nums1, nums2):
    common = set(nums1) & set(nums2)
    return max(common) if common else -1                   # SAI: đề yêu cầu nhỏ nhất

a, b = [1, 2, 3, 6], [2, 3, 4, 5]
print("đúng:", Solution().getCommon(a, b), "| sai:", sai_lay_max(a, b))
assert Solution().getCommon(a, b) != sai_lay_max(a, b)

đúng: 2 | sai: 3


**Vì sao sai:** đề hỏi giá trị chung **nhỏ nhất**. Chỉ cần đọc lại từ khóa "minimum" trong tên bài.

### Lỗi 2: Trả `0` thay vì `-1` khi không có giá trị chung

In [24]:
def sai_tra_0(nums1, nums2):
    common = set(nums1) & set(nums2)
    return min(common) if common else 0                    # SAI: phải là -1

a, b = [1, 2], [3, 4]
print("đúng:", Solution().getCommon(a, b), "| sai:", sai_tra_0(a, b))
assert Solution().getCommon(a, b) != sai_tra_0(a, b)

đúng: -1 | sai: 0


**Vì sao sai:** đề yêu cầu `-1` cho trường hợp không có giá trị chung. (Ngoài ra `min` của tập rỗng sẽ gây `ValueError` nếu không dùng `default`.)

### Lỗi 3: Tiến cả hai con trỏ khi chưa bằng nhau

In [25]:
def sai_tien_ca_hai(nums1, nums2):
    i = j = 0
    while i < len(nums1) and j < len(nums2):
        if nums1[i] == nums2[j]:
            return nums1[i]
        i += 1                                             # SAI: tiến cả hai
        j += 1
    return -1

a, b = [1, 2, 3], [2, 4]
print("đúng:", Solution().getCommon(a, b), "| sai:", sai_tien_ca_hai(a, b))
assert Solution().getCommon(a, b) != sai_tien_ca_hai(a, b)

đúng: 2 | sai: -1


**Vì sao sai:** chỉ được tiến con trỏ của mảng có phần tử **nhỏ hơn**. Tiến cả hai làm bỏ lỡ cặp `(a[1] = 2, b[0] = 2)`.

### Lỗi 4: Chỉ so sánh hai phần tử đầu mảng

In [26]:
def sai_chi_dau(nums1, nums2):
    return nums1[0] if nums1[0] == nums2[0] else -1        # SAI: chỉ xét phần tử đầu

a, b = [1, 2, 3], [2, 4]
print("đúng:", Solution().getCommon(a, b), "| sai:", sai_chi_dau(a, b))
assert Solution().getCommon(a, b) != sai_chi_dau(a, b)

đúng: 2 | sai: -1


**Vì sao sai:** giá trị chung có thể nằm ở bất kỳ vị trí nào. Hai mảng sắp xếp chỉ cho phép loại bỏ phần tử nhỏ hơn, không cho phép chỉ xét phần tử đầu.

### Lỗi 5: Dùng `in` trên `list` trong vòng lặp (chậm)

In [27]:
import time
def cham(nums1, nums2):
    for x in nums1:
        if x in nums2:                                    # x in list là O(m), tổng O(n * m)
            return x
    return -1

a = list(range(1, 20001, 2))
b = list(range(2, 20002, 2))
t0 = time.perf_counter(); r1 = cham(a, b); t1 = time.perf_counter()
r2 = Solution().getCommon(a, b); t2 = time.perf_counter()
assert r1 == r2 == -1
print(f"x in list: {(t1 - t0) * 1000:.1f} ms | hai con trỏ: {(t2 - t1) * 1000:.1f} ms")

x in list: 559.6 ms | hai con trỏ: 2.8 ms


**Cách tránh:** `x in list` duyệt tuyến tính. Chuyển mảng thành `set` (kiểm tra `O(1)`), hoặc dùng hai con trỏ / `bisect` tận dụng thứ tự sắp xếp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai mảng đã sắp xếp" + "tìm phần tử chung / hợp / giao" ⇒ **hai con trỏ** kiểu merge.
- Cần **giá trị chung nhỏ nhất** với mảng sắp xếp tăng ⇒ dừng ở giá trị chung **đầu tiên**.
- Không muốn nghĩ nhiều: `min(set(a) & set(b), default=-1)`.
- Một mảng rất ngắn, một mảng rất dài ⇒ duyệt mảng ngắn, `bisect` trong mảng dài.
- Nhớ đáp án mặc định `-1`, không phải `0`.
- Khi test, dùng mảng có phần tử lặp và mảng không có giá trị chung.

In [28]:
def one_line(a, b):
    return min(set(a) & set(b), default=-1)

assert one_line([1, 2, 3], [2, 4]) == 2 and one_line([1], [2]) == -1
print("phiên bản một dòng đúng")

phiên bản một dòng đúng


## 15. Biến thể của bài

1. **Tất cả giá trị chung** (giao của hai mảng, LeetCode 349/350): hai con trỏ nhưng không dừng khi gặp.
2. **Giao của ba mảng sắp xếp** (LeetCode 1213): ba con trỏ.
3. **Giá trị chung lớn nhất:** duyệt từ cuối hai mảng.
4. **Mảng chưa sắp xếp:** dùng `set` hoặc sắp xếp trước.
5. **Đếm số cặp `(i, j)` có `nums1[i] == nums2[j]`:** nhân số lần xuất hiện (cần `Counter`).
6. **Giá trị chung nhỏ nhất của `k` mảng:** dùng `heapq` hoặc đếm số mảng chứa từng giá trị.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt lồng nhau | so mọi cặp | `O(n * m)` | `O(1)` | chỉ để hiểu đề / mảng cực nhỏ |
| 2. Giao hai `set` | `set(a) & set(b)` rồi `min` | `O(n + m)` | `O(n + m)` | code ngắn, mảng chưa sắp xếp |
| 3. Hai con trỏ | tiến mảng có phần tử nhỏ hơn | `O(n + m)` | `O(1)` | cách nộp bài chuẩn |
| 4. `bisect` | duyệt mảng ngắn, tìm nhị phân | `O(min * log max)` | `O(1)` | một mảng ngắn hơn rất nhiều |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().getCommon([2, 4, 6], [1, 3, 6])` bằng bao nhiêu? Còn `Solution().getCommon([1, 5], [2, 3, 7])`?

**Bài 2 (tính tay).** Với `nums1 = [1, 4, 4, 9]` và `nums2 = [4, 5, 9, 10]`, chạy hai con trỏ bằng tay và ghi từng bước.

**Bài 3 (điền TODO).** Hoàn thành `common_all(nums1, nums2)` trả về **danh sách** các giá trị chung (không trùng, tăng dần) bằng hai con trỏ.

In [29]:
def common_all(nums1, nums2):
    res = []
    i = j = 0
    # TODO: hai con trỏ; khi bằng nhau thì thêm vào res (nếu khác phần tử cuối của res) rồi tiến cả hai
    return res

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `get_common_max(nums1, nums2)` trả về giá trị chung **lớn nhất** (hoặc `-1`) bằng hai con trỏ chạy từ **cuối** mỗi mảng.

**Bài 5 (biến thể / mini challenge).** Viết `get_common_three(a, b, c)` trả về giá trị chung nhỏ nhất của **ba** mảng đã sắp xếp (hoặc `-1`). Gợi ý: ba con trỏ; tiến con trỏ có giá trị nhỏ nhất.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2,4,6]` và `[1,3,6]` chung `6`; `[1,5]` và `[2,3,7]` không chung nên `-1`.
- **Mức 2 (vừa):** Bài 2: `(1,4)` tiến `i`; `(4,4)` bằng nhau nên đáp án `4`. Bài 3: sau khi bằng nhau, tiến cả `i` và `j`; kiểm tra `res` rỗng hoặc `res[-1] != x` trước khi thêm.
- **Mức 3 (sâu):** Bài 4: `i = len(a) - 1`, `j = len(b) - 1`; nếu `a[i] > b[j]` giảm `i`, ngược lại giảm `j`. Bài 5: nếu cả ba bằng nhau trả về; ngược lại tiến con trỏ có giá trị nhỏ nhất.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert Solution().getCommon([2, 4, 6], [1, 3, 6]) == 6
assert Solution().getCommon([1, 5], [2, 3, 7]) == -1

# Bài 2
assert Solution().getCommon([1, 4, 4, 9], [4, 5, 9, 10]) == 4

# Bài 3
def common_all(nums1, nums2):
    res = []
    i = j = 0
    while i < len(nums1) and j < len(nums2):
        if nums1[i] == nums2[j]:
            if not res or res[-1] != nums1[i]:
                res.append(nums1[i])
            i += 1
            j += 1
        elif nums1[i] < nums2[j]:
            i += 1
        else:
            j += 1
    return res

# Bài 4
def get_common_max(nums1, nums2):
    i, j = len(nums1) - 1, len(nums2) - 1
    while i >= 0 and j >= 0:
        if nums1[i] == nums2[j]:
            return nums1[i]
        if nums1[i] > nums2[j]:
            i -= 1
        else:
            j -= 1
    return -1

# Bài 5
def get_common_three(a, b, c):
    i = j = k = 0
    while i < len(a) and j < len(b) and k < len(c):
        if a[i] == b[j] == c[k]:
            return a[i]
        m = min(a[i], b[j], c[k])
        if a[i] == m:
            i += 1
        if b[j] == m:
            j += 1
        if c[k] == m:
            k += 1
    return -1

assert common_all([1, 4, 4, 9], [4, 5, 9, 10]) == [4, 9]
assert common_all([1, 2], [3, 4]) == []
assert common_all([2, 2, 2], [2, 2]) == [2]
assert get_common_max([1, 2, 3, 6], [2, 3, 4, 5]) == 3
assert get_common_max([1, 2], [3]) == -1
assert get_common_three([1, 2, 3], [2, 3, 4], [3, 4, 5]) == 3
assert get_common_three([1, 2], [2, 3], [4, 5]) == -1
random.seed(40)
for _ in range(300):
    a, b = gen(random.randint(1, 12), 1, 25), gen(random.randint(1, 12), 1, 25)
    assert common_all(a, b) == sorted(set(a) & set(b))
    c = gen(random.randint(1, 12), 1, 25)
    inter = set(a) & set(b) & set(c)
    assert get_common_three(a, b, c) == (min(inter) if inter else -1)
    cm = set(a) & set(b)
    assert get_common_max(a, b) == (max(cm) if cm else -1)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai con trỏ trên hai mảng đã sắp xếp)

```python
class Solution:
    def getCommon(self, nums1: List[int], nums2: List[int]) -> int:
        i = j = 0
        while i < len(nums1) and j < len(nums2):
            if nums1[i] == nums2[j]:
                return nums1[i]
            if nums1[i] < nums2[j]:
                i += 1
            else:
                j += 1
        return -1
```

**Checklist**

- [ ] Trả về giá trị chung **nhỏ nhất** (không phải lớn nhất)?
- [ ] Không có giá trị chung thì trả về `-1` (không phải `0`)?
- [ ] Chỉ tiến con trỏ của mảng có phần tử **nhỏ hơn**?
- [ ] Điều kiện vòng lặp `i < len(nums1) and j < len(nums2)` để không vượt chỉ số?
- [ ] Dùng `min(..., default=-1)` nếu giải bằng `set` để tránh `ValueError`?
- [ ] Đã thử mảng một phần tử, mảng không giao, mảng có phần tử lặp?
- [ ] Signature đúng `getCommon(self, nums1: List[int], nums2: List[int]) -> int`?